In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.nn import init
import os
import random
import pandas as pd

# Set seed for reproducibility
TORCH_SEED = 1337
torch.manual_seed(TORCH_SEED)

print("PyTorch Version:", torch.__version__)

PyTorch Version: 2.8.0+cpu


In [2]:
# Configuration
MODEL_PATH = 'sparse_moe_model.pt'
INPUT_FILE = '/content/tiny_shakespeare.txt'

# Hyperparameters
BATCH_SIZE = 16
BLOCK_SIZE = 32 # Context length
MAX_ITERS = 500
EVAL_INTERVAL = 100
LEARNING_RATE = 1e-3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EVAL_ITERS = 400
DROPOUT = 0.1

# Architecture Parameters
N_EMBED = 128
N_HEAD = 8
N_LAYER = 8
HEAD_SIZE = N_EMBED // N_HEAD

# Sparse MoE Parameters
NUM_EXPERTS = 8
TOP_K = 2
CAPACITY_FACTOR = 1.0 # Multiplier for expert capacity calculation

print(f"Running on device: {DEVICE}")
print(f"Model size: {N_LAYER} layers, {N_EMBED} dimensions.")

Running on device: cpu
Model size: 8 layers, 128 dimensions.


In [3]:
# Global variables for vocabulary and data
chars = []
vocab_size = 0
stoi = {}
itos = {}
encode = None
decode = None
train_data = None
val_data = None

def load_data_and_create_vocab():
    """
    Loads text data using pandas, creates vocabulary mappings, and splits data.
    Requires INPUT_FILE ('tinyshakespeare.txt') to be uploaded to Colab.
    """
    global chars, vocab_size, stoi, itos, encode, decode, train_data, val_data

    # Check for the file uploaded by the user
    if not os.path.exists(INPUT_FILE):
        raise FileNotFoundError(
            f"The input file '{INPUT_FILE}' was not found. Please upload it to your Colab session's file system."
        )

    # --- Load text using Pandas (as requested) ---
    try:
        # Read the file line by line into a DataFrame (assuming raw text separated by lines)
        # We set sep='\n' and header=None to treat each line as a row in the first column (0)
        df = pd.read_csv(INPUT_FILE, header=None, encoding='utf-8', sep='\n', on_bad_lines='skip', engine='python')
        # Combine all lines (rows) back into a single text string
        text = df[0].str.cat(sep='\n')
    except Exception as e:
        print(f"Error loading file with pandas: {e}. Falling back to standard Python open().")
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            text = f.read()
    # ---------------------------------------------

    chars = sorted(list(set(text)))
    vocab_size = len(chars)

    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for i, ch in enumerate(chars)}

    encode = lambda s: [stoi[c] for c in s]
    decode = lambda l: ''.join([itos[i] for i in l])

    data = torch.tensor(encode(text), dtype=torch.long)

    n = int(0.9 * len(data))
    train_data = data[:n]
    val_data = data[n:]

    print(f"Vocabulary size: {vocab_size} characters.")
    print(f"Training data size: {len(train_data)} tokens.")


def get_batch(split):
    """Generates a batch of inputs x and targets y."""
    data = train_data if split == 'train' else val_data
    if data is None:
        raise RuntimeError("Data not loaded. Call load_data_and_create_vocab() first.")

    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))

    x = torch.stack([data[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i + 1:i + BLOCK_SIZE + 1] for i in ix])

    x, y = x.to(DEVICE), y.to(DEVICE)
    return x, y

print(f"File preparation: Expecting '{INPUT_FILE}' to be uploaded manually.")

File preparation: Expecting '/content/tiny_shakespeare.txt' to be uploaded manually.


In [4]:
# From attention.py
class Head(nn.Module):
    """ One masked self-attention head. """

    def __init__(self):
        super().__init__()
        self.key = nn.Linear(N_EMBED, HEAD_SIZE, bias=False)
        self.query = nn.Linear(N_EMBED, HEAD_SIZE, bias=False)
        self.value = nn.Linear(N_EMBED, HEAD_SIZE, bias=False)
        # Causal mask buffer
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        # Compute attention scores (q @ k^T / sqrt(d_k))
        wei = q @ k.transpose(-2, -1) * (HEAD_SIZE ** -0.5)

        # Apply causal mask
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):

    def __init__(self):
        super().__init__()
        self.heads = nn.ModuleList([Head() for _ in range(N_HEAD)])
        self.proj = nn.Linear(N_EMBED, N_EMBED)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        # Concatenate outputs from all heads
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [5]:
# From experts.py
class Expert(nn.Module):
    """ A simple MLP that acts as an Expert in the MoE layer. """

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_EMBED, 4 * N_EMBED),
            nn.ReLU(),
            nn.Linear(4 * N_EMBED, N_EMBED),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.net(x)


class NoisyTopkRouter(nn.Module):
    """ Implements the Noisy Top-K Gating mechanism to select experts. """
    def __init__(self):
        super(NoisyTopkRouter, self).__init__()
        self.top_k = TOP_K
        self.num_experts = NUM_EXPERTS
        self.topkroute_linear = nn.Linear(N_EMBED, NUM_EXPERTS)
        self.noise_linear = nn.Linear(N_EMBED, NUM_EXPERTS)

    def forward(self, mh_output):
        logits = self.topkroute_linear(mh_output)
        noise_logits = self.noise_linear(mh_output)

        # Add scaled unit Gaussian noise
        noise = torch.randn_like(logits) * F.softplus(noise_logits)
        noisy_logits = logits + noise

        # Select the top-k experts
        top_k_logits, indices = noisy_logits.topk(self.top_k, dim=-1)

        # Create sparse tensor for softmax by setting unselected logits to -inf
        zeros = torch.full_like(noisy_logits, float('-inf'))
        sparse_logits = zeros.scatter(-1, indices, top_k_logits)

        router_output = F.softmax(sparse_logits, dim=-1)

        # router_output shape: [B, T, NUM_EXPERTS] (sparse with TOP_K non-zero values)
        # indices shape: [B, T, TOP_K] (indices of the selected experts)
        return router_output, indices

In [6]:
# From moe_block.py

class SparseMoE(nn.Module):
    """ The Sparse Mixture of Experts layer. """
    def __init__(self):
        super(SparseMoE, self).__init__()
        self.router = NoisyTopkRouter()
        self.experts = nn.ModuleList([Expert() for _ in range(NUM_EXPERTS)])
        self.top_k = TOP_K
        self.capacity_factor = CAPACITY_FACTOR
        self.num_experts = NUM_EXPERTS

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        # Get routing probabilities (gating_output) and selected expert indices
        gating_output, indices = self.router(x)

        # Flatten the input and gating outputs for easy indexing
        flat_x = x.view(-1, x.size(-1))
        flat_gating_output = gating_output.view(-1, gating_output.size(-1))

        # Calculate expert capacity
        tokens_per_batch = batch_size * seq_len * self.top_k
        expert_capacity = int((tokens_per_batch / self.num_experts) * self.capacity_factor)

        flat_indices = indices.view(-1, self.top_k)
        updates = torch.zeros_like(flat_x).to(x.device) # Ensure updates is on the correct device

        # Iterate over each expert
        for i, expert in enumerate(self.experts):
            # Identify which tokens are routed to expert 'i'
            expert_mask = (flat_indices == i)
            # Get the indices of the tokens (in the B*T space) routed to expert 'i'
            selected_indices = torch.nonzero(expert_mask.any(dim=-1)).squeeze(-1)

            # Apply Capacity Limit
            limited_indices = selected_indices[:expert_capacity]

            if limited_indices.numel() > 0:
                expert_input = flat_x[limited_indices]
                expert_output = expert(expert_input)

                # Get the routing score for the selected expert 'i'
                gating_scores = flat_gating_output[limited_indices, i].unsqueeze(1)
                weighted_output = expert_output * gating_scores

                # Accumulate weighted output back to the original token positions
                updates.index_add_(0, limited_indices, weighted_output)

        # Reshape and return
        final_output = updates.view(batch_size, seq_len, -1)

        return final_output


class Block(nn.Module):
    """ MoE Transformer Block: Self Attention followed by Sparse MoE (Pre-Norm). """

    def __init__(self):
        super().__init__()

        self.sa = MultiHeadAttention()
        self.smoe = SparseMoE()

        # Layer Norms
        self.ln1 = nn.LayerNorm(N_EMBED)
        self.ln2 = nn.LayerNorm(N_EMBED)

    def forward(self, x):
        # x = x + Attention(LayerNorm(x))
        x = x + self.sa(self.ln1(x))
        # x = x + MoE(LayerNorm(x))
        x = x + self.smoe(self.ln2(x))
        return x

In [7]:
# From model.py
class SparseMoELanguageModel(nn.Module):
    """ The main Sparse Mixture of Experts Language Model (GPT-style). """

    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size

        self.token_embedding_table = nn.Embedding(vocab_size, N_EMBED)
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, N_EMBED)

        # Stack of MoE Transformer Blocks
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])

        self.ln_f = nn.LayerNorm(N_EMBED)
        self.lm_head = nn.Linear(N_EMBED, vocab_size)

        self.apply(self._init_weights)


    def _init_weights(self, m):
        """Kaiming initialization for linear layers."""
        if isinstance(m, nn.Linear):
            init.kaiming_normal_(m.weight)
            if m.bias is not None:
                init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=DEVICE))

        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        """Generates new tokens based on the initial context (idx)."""
        for _ in range(max_new_tokens):
            # Crop idx to the last BLOCK_SIZE tokens for context
            idx_cond = idx[:, -BLOCK_SIZE:]

            # Get the predictions
            logits, _ = self(idx_cond)

            # Focus only on the last time step
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)

            # Append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [8]:
# From train.py

@torch.no_grad()
def estimate_loss(model):
    """Evaluates the model's loss on the train and validation datasets."""
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


def train():
    """Main training function for the Sparse MoE Language Model."""

    # 1. Load Data
    try:
        load_data_and_create_vocab()
    except FileNotFoundError as e:
        print(f"Error: {e}. Cannot start training.")
        return

    # 2. Initialize Model and Optimizer
    model = SparseMoELanguageModel(vocab_size)
    model = model.to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model initialized with {total_params / 1e6:.2f} Million parameters")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    # 3. Training Loop
    print("\n--- Starting Training ---")
    for iter_step in range(MAX_ITERS):

        if iter_step % EVAL_INTERVAL == 0 or iter_step == MAX_ITERS - 1:
            losses = estimate_loss(model)
            print(f"Step {iter_step}: Train loss {losses['train']:.4f}, Val loss {losses['val']:.4f}")

            # Save the model checkpoint
            torch.save(model.state_dict(), MODEL_PATH)

        xb, yb = get_batch('train')

        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    print("\nTraining finished.")
    torch.save(model.state_dict(), MODEL_PATH)
    print(f"Final model state saved to {MODEL_PATH}")

# Run Training
train()

Error loading file with pandas: Specified \n as separator or delimiter. This forces the python engine which does not accept a line terminator. Hence it is not allowed to use the line terminator as separator.. Falling back to standard Python open().
Vocabulary size: 65 characters.
Training data size: 1003853 tokens.
Model initialized with 9.00 Million parameters

--- Starting Training ---
Step 0: Train loss 5.2955, Val loss 5.2835
Step 100: Train loss 2.8746, Val loss 2.8937
Step 200: Train loss 2.5961, Val loss 2.5974
Step 300: Train loss 2.5001, Val loss 2.5028
Step 400: Train loss 2.4249, Val loss 2.4154
Step 499: Train loss 2.3565, Val loss 2.3692

Training finished.
Final model state saved to sparse_moe_model.pt


In [9]:
# From generate.py

def generate_text(start_string, max_tokens=300):
    """Loads the trained model and generates text based on a starting string."""

    # Re-run data load to ensure vocab is current (should be okay if train ran)
    try:
        load_data_and_create_vocab()
    except FileNotFoundError:
        print("Error: Cannot generate text without a vocabulary.")
        return

    if not os.path.exists(MODEL_PATH):
        print(f"Error: Model file '{MODEL_PATH}' not found. Did training complete successfully?")
        return

    # Initialize model structure and load weights
    model = SparseMoELanguageModel(vocab_size)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

    model.to(DEVICE)
    model.eval()

    # Input handling
    if not all(c in stoi for c in start_string):
        # Fallback for unknown characters
        known_chars = list(stoi.keys())
        known_char = known_chars[0] if known_chars else ' '
        print(f"Warning: Starting string contains unknown characters. Replacing with '{known_char}'.")
        start_string = ''.join([c if c in stoi else known_char for c in start_string])

    context = torch.tensor(encode(start_string), dtype=torch.long, device=DEVICE).unsqueeze(0)

    print("\n--- Generated Text ---")

    # Generate the tokens
    generated_indices = model.generate(context, max_tokens)
    generated_text = decode(generated_indices[0].tolist())

    print(generated_text)
    print("----------------------")


# Run Generation
prompt = "The sparse mixture"
print(f"Generating text starting with: '{prompt}'")
generate_text(prompt, max_tokens=300)

Generating text starting with: 'The sparse mixture'
Error loading file with pandas: Specified \n as separator or delimiter. This forces the python engine which does not accept a line terminator. Hence it is not allowed to use the line terminator as separator.. Falling back to standard Python open().
Vocabulary size: 65 characters.
Training data size: 1003853 tokens.

--- Generated Text ---
The sparse mixtureiceng'e.
AUCEENULEE:,S: SuwV, lelese tir the do thick alest hecst n  ete fastlifres
Ve sherbigatld
The.
UAK: pK, whedmy yell.t hooven mid abe to an ble
Ayan ge yo na fowktns, spnvan.
N 'se t erat seyoulead-s youee: she
ches as ssthactlly heace dink-nd mm' 'y
Yl ad e o thucens
As neeks stoe ank las.

----------------------
